In [1]:
import warnings
warnings.filterwarnings("ignore", category=UserWarning, module=r"icdn(\.|$)")
warnings.filterwarnings("ignore", category=FutureWarning)

In [ ]:
from pathlib import Path
import pandas as pd
import json
from icdn import ICDNModel, ICDNConfig, PanelSchema

ROOT = Path.cwd() if Path.cwd().name != "notebooks" else Path.cwd().parent

DATASETS = {
    "walmart": {
        "path": ROOT / "data" / "M5-walmart" / "panel" / "m5_icdn_panel.parquet",
        "out": ROOT / "data" / "M5-walmart" / "panel" / "icdn",
        "hp": ROOT / "data" / "M5-walmart" / "panel" / "icdn-opt" / "best_params.json",
        "schema": PanelSchema(category="category"),
        "own_elasticity_bounds": (-3.5, 0.0),
        "cross_elasticity_bounds": (-0.4, 0.8),
        "beta_prior": -2.0,
        "k_neighbors": 4,
        "same_category_first": True,
        "min_coverage": 0.5,
    },
    "one_c": {
        "path": ROOT / "data" / "predict-future-sales-1c" / "panel" / "1c_icdn_panel.parquet",
        "out": ROOT / "data" / "predict-future-sales-1c" / "panel" / "icdn",
        "hp": ROOT / "data" / "predict-future-sales-1c" / "panel" / "icdn-opt" / "best_params.json",
        "schema": PanelSchema(category="category"),
        "own_elasticity_bounds": (-3.0, 0.0),
        "cross_elasticity_bounds": (-0.2, 0.5),
        "beta_prior": -1.5,
        "k_neighbors": 3,
        "same_category_first": True,
        "min_coverage": 0.15,
    },
}

In [ ]:
def load_hp(spec):
    path = spec.get("hp")
    if path is None:
        return None
    if not path.exists():
        raise FileNotFoundError(f"run icdn-opt.ipynb first: {path}")
    hp = json.loads(path.read_text())
    hp["hidden"] = tuple(hp["hidden"])
    print("  hp:", hp)
    return hp

def fit_one(name: str, spec: dict) -> None:
    panel = pd.read_parquet(spec["path"])
    panel = panel[panel["price"] > 0].copy()
    n_products = int(panel["product_code"].nunique())
    print(f"\n=== {name} === rows={len(panel):,}  products={n_products}  "
          f"stores={panel['store_code'].nunique()}  weeks={panel['week_id'].nunique()}")
    hp = load_hp(spec)
    searched = {
        "hidden": hp["hidden"] if hp else (256, 128, 64),
        "dropout": hp["dropout"] if hp else 0.2547,
        "lr": hp["lr"] if hp else 1.6246e-3,
        "warmup_lr": hp["warmup_lr"] if hp else 1.6856e-3,
        "lambda_smooth": hp["lambda_smooth"] if hp else 0.0351,
        "lambda_elast": hp["lambda_elast"] if hp else 0.0445,
        "k_neighbors": hp["k_neighbors"] if hp else spec["k_neighbors"],
        "n_knots": hp["n_knots"] if hp else 3,
    }
    cfg = ICDNConfig(
        schema=spec["schema"],
        n_products=n_products,
        k_neighbors=min(int(searched["k_neighbors"]), max(n_products - 1, 1)),
        same_category_first=spec["same_category_first"],
        own_elasticity_bounds=spec["own_elasticity_bounds"],
        cross_elasticity_bounds=spec["cross_elasticity_bounds"],
        beta_prior=spec["beta_prior"],
        min_coverage=spec["min_coverage"],
        hidden=searched["hidden"],
        dropout=searched["dropout"],
        lr=searched["lr"],
        warmup_lr=searched["warmup_lr"],
        lambda_smooth=searched["lambda_smooth"],
        lambda_elast=searched["lambda_elast"],
        n_knots=int(searched["n_knots"]),
        enforce_negative_beta=True,
        warmup_epochs=25,
        epochs=50,
        early_stopping_patience=12,
        seed=42,
        verbose=True,
    )
    model = ICDNModel(cfg)
    model.fit(panel)

    elast = model.elasticities()
    scored = model.score(panel)
    out = spec["out"]
    out.mkdir(parents=True, exist_ok=True)
    model.save(out / "model")
    elast.to_csv(out / "elasticities.csv", index=False)
    scored.to_csv(out / "scores.csv", index=False)

    own = elast.loc[elast["kind"] == "own", "elasticity"]
    cross = elast.loc[elast["kind"] == "cross", "elasticity"]
    print(elast.head(12))
    print("own   mean/min/max:", float(own.mean()), float(own.min()), float(own.max()))
    print("cross mean/min/max:", float(cross.mean()), float(cross.min()), float(cross.max()))

for name, spec in DATASETS.items():
    fit_one(name, spec)


=== dunnhumby === rows=3,216  products=10  stores=20  weeks=102
[warmup] epoch 1/25 train=0.5780 val=0.3993
[warmup] epoch 11/25 train=0.1001 val=0.1098
[warmup] epoch 21/25 train=0.0776 val=0.0768
[warmup] epoch 25/25 train=0.0656 val=0.0732
[main] epoch 1/50 train=0.0000 val=0.1841
[main] epoch 11/50 train=0.4470 val=1.1647
[main] early stop at epoch 13
   store_code product_code competitor   kind  elasticity       std  \
0         292      1010190    1010190    own   -0.063875  0.023849   
1         292      1010190    1053690  cross    0.046419  0.018275   
2         292      1010190    1076875  cross    0.017335  0.011193   
3         292      1010190    1092026  cross    0.030891  0.004548   
4         292      1053690    1010190  cross   -0.057050  0.080320   
5         292      1053690    1053690    own   -0.173482  0.155873   
6         292      1053690    1076875  cross   -0.094888  0.119144   
7         292      1053690    1092026  cross   -0.041049  0.083718   
8         2